In [1]:
# setup cv
import os
os.environ["OPENCV_VIDEOIO_MSMF_ENABLE_HW_TRANSFORMS"] = "0"
os.environ["MUJOCO_GL"] = "glfw"

# dataset
from lerobot.datasets.transforms import ImageTransformsConfig
from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata
from huggingface_hub import HfApi

# policy
from lerobot.policies.act.configuration_act import ACTConfig

# lerobot training
from lerobot.configs.train import TrainPipelineConfig, DatasetConfig
from lerobot.optim.optimizers import AdamWConfig
from lerobot.configs.default import WandBConfig, EvalConfig
from lerobot.utils.utils import init_logging
from lerobot.scripts import train

# torch
from torch.cuda import is_available

# wandb
import wandb

# utils
from dotenv import load_dotenv
from pprint import pprint
import sys

# my code
from src.paths import REPO_ROOT, DATASETS_DIR, HF_NAME, POLICIES_DIR

# set up env secrets
load_dotenv(REPO_ROOT / ".env", override=True)


True

# Must be run as admin!

In [2]:
device = "cuda" if is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [3]:
PUSH_TO_HUB = False
REPO_NAME = 'so101_leg_pick_and_place'
EXPERIMENT_NAME = '50_episodes_v0'
resume_checkpoint = None
bad_eps = {10, 11, 22}

In [4]:
DATASET_PATH = DATASETS_DIR / REPO_NAME
DATASET_ID = f"{HF_NAME}/{REPO_NAME}"

In [5]:
if PUSH_TO_HUB:
    api = HfApi(token=os.getenv("HUGGINGFACE_TOKEN"))
    assert HF_NAME == api.whoami()['name']

In [6]:
wandb.login(key = os.getenv('WANDB_API_KEY'))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\Jonathan\_netrc
wandb: Currently logged in as: jonathm126 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [7]:
ds_meta = LeRobotDatasetMetadata(DATASET_ID, root = DATASET_PATH)

In [8]:
print(f"Total number of episodes: {ds_meta.total_episodes}")
print(f"Average number of frames per episode: {ds_meta.total_frames / ds_meta.total_episodes:.3f}")
print(f"Frames per second used during data collection: {ds_meta.fps}")
print(f"Robot type: {ds_meta.robot_type}")
print(f"keys to access images from cameras: {ds_meta.camera_keys=}\n")

Total number of episodes: 47
Average number of frames per episode: 801.383
Frames per second used during data collection: 30
Robot type: so101_follower
keys to access images from cameras: ds_meta.camera_keys=['observation.images.wrist_cam', 'observation.images.top_cam']



In [9]:
print("Features:")
pprint(ds_meta.features.keys())

Features:
dict_keys(['action', 'observation.state', 'observation.images.wrist_cam', 'observation.images.top_cam', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index'])


In [10]:
# print('observation.images.wrist_cam')
# pprint(ds_meta.features['observation.images.wrist_cam'])

# print('observation.images.top_cam')
# pprint(ds_meta.features['observation.images.top_cam'])

print('observation.state')
pprint(ds_meta.features['observation.state'])

observation.state
{'dtype': 'float32',
 'names': ['shoulder_pan.pos',
           'shoulder_lift.pos',
           'elbow_flex.pos',
           'wrist_flex.pos',
           'wrist_roll.pos',
           'gripper.pos'],
 'shape': (6,)}


In [11]:
print('Action Space:')
pprint(ds_meta.features['action'])

Action Space:
{'dtype': 'float32',
 'names': ['shoulder_pan.pos',
           'shoulder_lift.pos',
           'elbow_flex.pos',
           'wrist_flex.pos',
           'wrist_roll.pos',
           'gripper.pos'],
 'shape': (6,)}


In [12]:
# select image transforms
transforms_cfg = ImageTransformsConfig(enable = True,
                                        max_num_transforms = 3,
                                        )

# select episodes
if bad_eps:
    episodes = [ep for ep in range(len(ds_meta.episodes)) if ep not in bad_eps]

# build dataset
dataset_cfg = DatasetConfig(repo_id = DATASET_ID,
                            root               = DATASET_PATH,   # dataset location
                            episodes           = episodes,       # which episodes to use
                            image_transforms   = transforms_cfg, # configure image transformations
                            use_imagenet_stats = True,           # normalize image using imagenet weights
                            video_backend      = 'pyav'          # for windows
                            )

Policy Config

In [13]:
input_features = {
    "observation.images.wrist_cam": {
        "type" : "VISUAL",
        "shape": [3, 480, 640]
    },
    "observation.images.top_cam": {
        "type" : "VISUAL",
        "shape": [3, 480, 640]
    },
    "observation.state": {
        "type" : "STATE",
        "shape": [6]
    }
}

output_features = {
    "action": {
        "type" : "ACTION",
        "shape": [6]
    }
}

In [14]:
policy_config = ACTConfig(n_obs_steps = 1, # how many observations does the policy need
                            chunk_size              = 50,             # action chunking size
                            n_action_steps          = 50,               # how many actions to preform (?) TODO - reduce to 10
                            device                  = device,
                            input_features          = input_features,
                            output_features         = output_features,
                            push_to_hub             = False,
                            temporal_ensemble_coeff = None,
                            use_vae                 = True
                            )

Training params config:

In [15]:
optimizer_cfg = AdamWConfig(lr = 1e-5, weight_decay = 1e-4)

WandB logging:

In [16]:
wandb_config = WandBConfig(enable = True, 
                            disable_artifact = False,
                            project          = 'ACT-Real-Sim-Real',
                            entity           = 'jonathm126-personal',
                            mode             = 'online',
                            run_id           =  EXPERIMENT_NAME + '-train' ) # careful - this is a unique ids
print(wandb_config.run_id)

50_episodes_v0-train


Integration:

In [ ]:
train_cfg = TrainPipelineConfig(dataset                    = dataset_cfg,
                                # env                        = env_config,                 # sim env
                                policy                     = policy_config,              # policy config
                                output_dir                 = POLICIES_DIR / policy_config.type / REPO_NAME / EXPERIMENT_NAME,
                                job_name                   = REPO_NAME + '-' + EXPERIMENT_NAME + '-train', # name
                                use_policy_training_preset = True,
                                optimizer                  = optimizer_cfg,
                                wandb                      = wandb_config,
                                steps                      = 100000,
                                log_freq                   = 200,
                                # eval_freq                  = 20000,
                                save_checkpoint            = True,
                                save_freq                  = 20000,
                                batch_size                 = 6
                                )

Train:

In [ ]:
if resume_checkpoint is None:
    init_logging()
    train.train(train_cfg)
else:
    assert resume_checkpoint is not None
    pth = POLICIES_DIR / policy_config.type / REPO_NAME / EXPERIMENT_NAME / 'checkpoints' / resume_checkpoint / 'pretrained_model' / 'train_config.json'
    assert pth.exists()
    sys.argv = [
        "train",
        f"--config_path={pth}",
        "--resume=true",
    ]
    init_logging()
    train.train()

INFO 2025-09-19 17:33:42 ts\train.py:111 {'batch_size': 8,
 'dataset': {'episodes': [0,
                          1,
                          2,
                          3,
                          4,
                          5,
                          6,
                          7,
                          8,
                          9,
                          12,
                          13,
                          14,
                          15,
                          16,
                          17,
                          18,
                          19,
                          20,
                          21,
                          23,
                          24,
                          25,
                          26,
                          27,
                          28,
                          29,
                          30,
                          31,
                          32,
                          33,
                     

wandb: WARNING The get_url method is deprecated and will be removed in a future release. Please use `run.url` instead.
INFO 2025-09-19 17:33:43 db_utils.py:103 Track this run --> https://wandb.ai/jonathm126-personal/ACT-Real-Sim-Real/runs/50_episodes_v0-train
INFO 2025-09-19 17:33:43 ts\train.py:127 Creating dataset


Logs will be synced with wandb.


Resolving data files:   0%|          | 0/44 [00:00<?, ?it/s]

INFO 2025-09-19 17:33:44 ts\train.py:138 Creating policy
INFO 2025-09-19 17:33:44 ts\train.py:144 Creating optimizer and scheduler
INFO 2025-09-19 17:33:44 ts\train.py:156 Output dir: D:\Github\IDC_Project-Sim_Real_Sim\policies\act\so101_leg_pick_and_place\50_episodes_v0
INFO 2025-09-19 17:33:44 ts\train.py:159 cfg.steps=100000 (100K)
INFO 2025-09-19 17:33:44 ts\train.py:160 dataset.num_frames=29599 (30K)
INFO 2025-09-19 17:33:44 ts\train.py:161 dataset.num_episodes=44
INFO 2025-09-19 17:33:44 ts\train.py:162 num_learnable_params=51571590 (52M)
INFO 2025-09-19 17:33:44 ts\train.py:163 num_total_params=51571638 (52M)
INFO 2025-09-19 17:33:44 ts\train.py:202 Start offline training on a fixed dataset
INFO 2025-09-19 17:34:58 ts\train.py:232 step:200 smpl:2K ep:2 epch:0.05 loss:7.118 grdn:163.629 lr:1.0e-05 updt_s:0.253 data_s:0.114
INFO 2025-09-19 17:35:49 ts\train.py:232 step:400 smpl:3K ep:5 epch:0.11 loss:2.908 grdn:90.410 lr:1.0e-05 updt_s:0.253 data_s:0.002
INFO 2025-09-19 17:36:38 t